# 00. 환경 세팅 확인 노트북

각자 환경이 다르면 같은 코드가 다르게 돈다. 이 노트북을 위에서부터 끝까지 실행해서
**전부 통과하면 준비 끝**. 에러가 나면 그 셀의 출력을 카톡에 올릴 것.

사전 준비:
```bash
python3.11 -m venv .venv
source .venv/bin/activate   # Windows: .venv\Scripts\activate
pip install -r requirements.txt
```
QGIS도 따로 설치해 둔다 (좌표계 어긋났을 때 눈으로 확인하는 게 가장 빠르다).

In [ ]:
# 1. 파이썬 버전 확인 — 3.11.x 여야 한다
import sys
print(sys.version)
assert sys.version_info[:2] == (3, 11), "python 3.11이 아니다! venv를 3.11로 다시 만들 것"
print("OK: python 3.11")

In [ ]:
# 2. 패키지 임포트 + 버전 확인 — requirements.txt 고정 버전과 일치해야 한다
import importlib

packages = {
    "geopandas": "1.1.1", "osmnx": "2.0.3", "networkx": "3.4.2",
    "rasterio": "1.4.3", "rioxarray": "0.18.2", "shapely": "2.1.1",
    "pyproj": "3.7.1", "pandas": "2.2.3", "numpy": "2.1.3",
    "sklearn": "1.6.1", "scipy": "1.15.2", "matplotlib": "3.10.0",
    "folium": "0.19.4", "contextily": "1.6.2", "mapclassify": "2.8.1",
    "requests": "2.32.3", "tqdm": "4.67.1",
}
bad = []
for name, want in packages.items():
    try:
        mod = importlib.import_module(name)
        got = getattr(mod, "__version__", "?")
        mark = "OK " if got == want else "!! "
        if got != want:
            bad.append(name)
        print(f"{mark}{name:<12} {got:<10} (기준 {want})")
    except ImportError as e:
        bad.append(name)
        print(f"XX {name:<12} 임포트 실패: {e}")
if bad:
    print(f"\n버전이 어긋난 패키지: {bad} → pip install -r requirements.txt 다시 실행")
else:
    print("\n전부 일치. OK")

In [ ]:
# 3. 좌표계 테스트 — 프로젝트 표준 EPSG:5187 (동부원점 TM)
# 부산시청(위경도)을 5187로 변환해서 상식적인 범위에 들어오는지 확인한다
from pyproj import Transformer

lon, lat = 129.0756, 35.1799   # 부산시청 근처
tf_5187 = Transformer.from_crs("EPSG:4326", "EPSG:5187", always_xy=True)
tf_5179 = Transformer.from_crs("EPSG:4326", "EPSG:5179", always_xy=True)
x87, y87 = tf_5187.transform(lon, lat)
x79, y79 = tf_5179.transform(lon, lat)
print(f"부산시청  EPSG:5187 → x={x87:,.1f}, y={y87:,.1f}")
print(f"부산시청  EPSG:5179 → x={x79:,.1f}, y={y79:,.1f}")
# 5187: 동부원점(129E) 기준이라 부산의 x는 원점값 200,000 부근이어야 정상
assert 150_000 < x87 < 250_000, "x가 이상하다 — 중부원점(5186)을 쓰고 있는 건 아닌지 확인"
print("OK: 동부원점 TM 변환 정상 (부산은 5186이 아니라 5187!)")

In [ ]:
# 4. GeoDataFrame + 지도 렌더링 테스트
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

gdf = gpd.GeoDataFrame(
    {"name": ["부산시청", "서면", "해운대해수욕장"]},
    geometry=[Point(129.0756, 35.1799), Point(129.0595, 35.1578), Point(129.1603, 35.1587)],
    crs="EPSG:4326",
).to_crs("EPSG:5187")
print(gdf)

ax = gdf.plot(figsize=(5, 5), color="red")
ax.set_title("EPSG:5187 test plot")
plt.show()
print("OK: geopandas + matplotlib 정상")

In [ ]:
# 5. folium 인터랙티브 지도 테스트 (셀 아래에 지도가 뜨면 OK)
import folium

m = folium.Map(location=[35.1799, 129.0756], zoom_start=12)
folium.Marker([35.1799, 129.0756], tooltip="부산시청").add_to(m)
m

In [ ]:
# 6. rasterio / DEM 처리 준비 확인 (GDAL 바인딩이 제대로 붙었는지)
import rasterio
import numpy as np

print("rasterio", rasterio.__version__, "| GDAL", rasterio.__gdal_version__)
# 메모리 상에서 작은 래스터를 만들고 읽어본다
from rasterio.io import MemoryFile
arr = np.random.rand(1, 10, 10).astype("float32")
with MemoryFile() as mem:
    with mem.open(driver="GTiff", height=10, width=10, count=1,
                  dtype="float32", crs="EPSG:5187",
                  transform=rasterio.transform.from_origin(200000, 300000, 10, 10)) as ds:
        ds.write(arr)
    with mem.open() as ds:
        assert ds.read().shape == (1, 10, 10)
print("OK: rasterio 읽기/쓰기 정상")

## 자주 나는 문제

| 증상 | 처방 |
|------|------|
| `rasterio`/`geopandas` 설치 실패 (Windows) | `pip`을 최신으로 올리고 재시도. 그래도 안 되면 conda-forge로 해당 패키지만 설치 |
| 한글 폰트 깨짐 (matplotlib) | Windows: `plt.rc('font', family='Malgun Gothic')` / macOS: `AppleGothic` / 리눅스: 나눔폰트 설치 |
| 3번 셀 assert 실패 | EPSG:5186(중부원점)을 쓰고 있을 가능성. 부산은 **5187(동부원점)** |
| folium 지도가 안 뜸 | 노트북 신뢰 설정(Trust Notebook) 후 재실행 |

전부 통과했으면 카톡에 "환경 OK" 한 줄 올리기.